# Load Keystone Eval Results from S3

Loads every `eval_result.json` under an S3 prefix, validates each as a
`KeystoneRepoResult`, and flattens into a single DataFrame.

---

> **Note:** This notebook is checked in without cell outputs.  
> To execute and save results to a separate file, run:
> ```bash
> uv run jupyter nbconvert --to notebook --execute evals/eda/load_s3_eval_results.ipynb --output load_s3_eval_results_executed.ipynb
> ```

In [ ]:
import json

import pandas as pd
import s3fs
from pydantic import ValidationError

from evals.eval_schema import KeystoneRepoResult

In [ ]:
S3_PREFIX = "s3://int8-datasets/keystone/evals/2026-03-08_four_model_thad_v2"

fs = s3fs.S3FileSystem(anon=False)

# Strip s3:// for s3fs path operations
prefix_path = S3_PREFIX.removeprefix("s3://")
json_paths = fs.glob(f"{prefix_path}/**/eval_result.json")
print(f"Found {len(json_paths)} eval_result.json files")

In [ ]:
results: list[KeystoneRepoResult] = []
errors: list[dict] = []

for s3_path in json_paths:
    with fs.open(s3_path, "r") as f:
        raw = json.load(f)
    try:
        result = KeystoneRepoResult(**raw)
        results.append(result)
    except ValidationError as e:
        errors.append({"path": s3_path, "error": str(e)})

print(f"Validated {len(results)} results, {len(errors)} validation errors")
if errors:
    for err in errors[:5]:
        print(f"  FAIL: {err['path']}")
        print(f"        {err['error'][:200]}")

In [ ]:
records = []
for r in results:
    br = r.bootstrap_result
    agent = br.agent if br else None
    ver = br.verification if br else None
    cost = agent.cost if agent else None

    # Extract config name from the S3 path (first directory after prefix)
    config_name = None
    if r.keystone_config and r.keystone_config.agent_config:
        # Use model as a proxy for config name
        model = r.keystone_config.agent_config.model
        config_name = model.value if model else None

    records.append(
        {
            "repo_id": r.repo_entry.id,
            "config_name": config_name,
            "trial_index": r.trial_index,
            "success": r.success,
            "error_message": r.error_message,
            # Agent execution
            "agent_exit_code": agent.exit_code if agent else None,
            "agent_walltime_seconds": agent.duration_seconds if agent else None,
            "agent_timed_out": agent.timed_out if agent else None,
            # Keystone exit code: success maps to 0, failure to 1
            "keystone_exit_code": 0 if r.success else 1,
            # Cost
            "cost_usd": cost.cost_usd if cost else None,
            "input_tokens": cost.token_spending.input if cost else None,
            "output_tokens": cost.token_spending.output if cost else None,
            # Verification
            "image_build_seconds": ver.image_build_seconds if ver else None,
            "test_execution_seconds": ver.test_execution_seconds if ver else None,
            "tests_passed": ver.tests_passed if ver else None,
            "tests_failed": ver.tests_failed if ver else None,
            # Agent status messages
            "summary": agent.summary.message if agent and agent.summary else None,
            "status_messages": [
                {"timestamp": sm.timestamp, "message": sm.message}
                for sm in (agent.status_messages if agent else [])
            ],
        }
    )

df = pd.DataFrame(records)
print(f"DataFrame: {len(df)} rows, {len(df.columns)} columns")
df.head()

In [ ]:
df.info()

In [ ]:
# Quick summary by config
if "config_name" in df.columns and df["config_name"].notna().any():
    summary = (
        df.groupby("config_name")
        .agg(
            total=("success", "count"),
            successes=("success", "sum"),
            success_rate=("success", "mean"),
            mean_cost_usd=("cost_usd", "mean"),
            mean_agent_walltime=("agent_walltime_seconds", "mean"),
            mean_tests_passed=("tests_passed", "mean"),
            mean_tests_failed=("tests_failed", "mean"),
        )
    )
    summary["success_rate"] = (summary["success_rate"] * 100).round(1)
    summary